# 03 · Config 1: LLM sem contexto (e 1b, apêndice)

- **Config 1 (oficial):** só o LLM local, sem consultar a base, **sem** instrução de recusa e sem mencionar base de conhecimento. Mesmo JSON de saída.
- **Config 1b (apêndice):** igual, mas com a instrução "se não souber, recuse".

Os resultados vão para `avaliacao/resultados/sem_contexto.jsonl` e `sem_contexto_instr.jsonl`, de forma incremental (retoma se for interrompido).

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src')); sys.path.insert(0, str(RAIZ / 'avaliacao'))

import avaliar as av
from sonar_ia import configs
from sonar_ia.llm import nome_modelo

casos = av.carregar_casos()
print('modelo LLM:', nome_modelo(), '| casos:', len(casos))

modelo LLM: qwen2.5:3b | casos: 63


In [2]:
reg1 = av.avaliar('sem_contexto', configs.sem_contexto, casos, verbose=False)
reg1b = av.avaliar('sem_contexto_instr', configs.sem_contexto_instr, casos, verbose=False)
print(len(reg1), 'casos na config 1;', len(reg1b), 'casos na config 1b')

63 casos na config 1; 63 casos na config 1b


## Métricas lado a lado

In [3]:
nomes = {'1 (oficial)': reg1, '1b (com instrução)': reg1b}
print('acuracia/alucinacao: sobre todos os casos do grupo; recusa_correta: só nao_consta;')
print('falsa_recusa: casos com resposta (texto e rota com linha); citacao_correta: só texto\n')
print('GERAL'); print(pd.DataFrame({k: av.resumo(v) for k, v in nomes.items()}).T.to_string())
for tipo in ('texto', 'rota', 'nao_consta'):
    print(f'\n{tipo.upper()}')
    print(pd.DataFrame({k: av.resumo([r for r in v if r['tipo'] == tipo]) for k, v in nomes.items()}).T.to_string())

acuracia/alucinacao: sobre todos os casos do grupo; recusa_correta: só nao_consta;
falsa_recusa: casos com resposta (texto e rota com linha); citacao_correta: só texto

GERAL
                       n  acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
1 (oficial)         63.0    0.2381      0.7619          0.0667        0.0000          1.0              0.0              0.74
1b (com instrução)  63.0    0.4603      0.2698          0.9333        0.4048          1.0              0.0              0.63

TEXTO
                       n  acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
1 (oficial)         30.0    0.3667      0.6333             NaN        0.0000          1.0              0.0              0.76
1b (com instrução)  30.0    0.3000      0.3667             NaN        0.3333          1.0              0.0              0.67

ROTA
                       n  acuracia  alucinacao  recusa_correta

## Exemplos e erros da config 1 (oficial)

In [4]:
def mostrar(r):
    s = r['saida']
    print(f"{r['id']} ({r['tipo']}) {r['latencia_s']}s  json_valido={s['json_valido']}")
    print('  pergunta :', r['pergunta'])
    print('  resposta :', s['resposta'])
    print('  fonte    :', s['fonte'], '| confianca:', s['confianca'], '| recusou:', s['recusou'])
    print('  métricas :', {k: v for k, v in r['metricas'].items() if v})

alu = [r for r in reg1 if r['metricas']['alucinacao']]
print(f'Alucinações da config 1: {len(alu)} de {len(reg1)}')
for tipo in ('texto', 'rota', 'nao_consta'):
    print(f'  {tipo}:', sum(r['tipo'] == tipo for r in alu), 'de', sum(r['tipo'] == tipo for r in reg1))
print()
for r in [x for x in alu if x['tipo'] == 'nao_consta'][:3] + [x for x in alu if x['tipo'] == 'texto'][:2]:
    mostrar(r); print()
print('Acertos da config 1 em casos que exigiam a base (texto):', [r['id'] for r in reg1 if r['tipo'] == 'texto' and r['metricas']['acerto']])

Alucinações da config 1: 48 de 63
  texto: 19 de 30
  rota: 15 de 18
  nao_consta: 14 de 15

nao_consta_01 (nao_consta) 0.69s  json_valido=True
  pergunta : a estação faria lima da linha 4 amarela tem elevador?
  resposta : Não, a estação Lima da Linha 4-Amarela não possui elevador.
  fonte    : [] | confianca: 1.0 | recusou: False
  métricas : {'json_valido': True, 'alucinacao': True}

nao_consta_02 (nao_consta) 0.7s  json_valido=True
  pergunta : tem elevador na estação adolfo pinheiro da linha 5 lilás?
  resposta : Não há elevador na estação Adolfo Pinheiro da linha 5 Lilás.
  fonte    : [] | confianca: 1.0 | recusou: False
  métricas : {'json_valido': True, 'alucinacao': True}

nao_consta_03 (nao_consta) 0.64s  json_valido=True
  pergunta : a estaçao carapicuiba da linha 8 diamante é acessivel pra cego?
  resposta : Não, esta estação não é acessível para pessoas cegas.
  fonte    : [] | confianca: 1.0 | recusou: False
  métricas : {'json_valido': True, 'alucinacao': True}

texto_02

## Config 1b: o que a instrução de recusa muda

In [5]:
rec = [r for r in reg1b if r['metricas']['recusou']]
print(f'1b recusou {len(rec)} de {len(reg1b)} casos')
print('  por tipo:', {t: sum(r['tipo'] == t for r in rec) for t in ('texto', 'rota', 'nao_consta')})
print('  falsas recusas (respondíveis recusados):', sum(r['metricas']['falsa_recusa'] for r in reg1b))
print('  alucinações:', sum(r['metricas']['alucinacao'] for r in reg1b))
print('1 (oficial) recusou', sum(r['metricas']['recusou'] for r in reg1), 'de', len(reg1))

1b recusou 37 de 63 casos
  por tipo: {'texto': 10, 'rota': 13, 'nao_consta': 14}
  falsas recusas (respondíveis recusados): 17
  alucinações: 17
1 (oficial) recusou 1 de 63
